# ADA 442 Statistical Learning Final Project

## Bank Term Deposit Subscription Prediction

This notebook builds a binary classification model for the Bank Marketing dataset. The goal is to predict whether a bank client subscribes to a term deposit. The target column is `y`, where `yes` means the client subscribed and `no` means the client did not subscribe.

The notebook follows the same general structure as the Streamlit app and prepares a final pipeline that can be saved as `rf_pipeline.sav`.


## 1. Import Libraries

First, I import the main libraries for data loading, preprocessing, modeling, evaluation, and saving the final pipeline.


In [1]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load the Dataset

The notebook is designed to run from inside the `ADA-PROJECT-FINAL` folder. The dataset path is relative to that folder.


In [2]:
data_path = Path("bank-additional") / "bank-additional.csv"
df = pd.read_csv(data_path, sep=";")
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


## 3. Basic Dataset Information

Here I check the dataset size, preview the first rows, list the columns, check missing values, and review the target distribution.


In [3]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values by column:")
print(df.isna().sum())

print("\nTarget class distribution:")
print(df["y"].value_counts())
print("\nTarget class distribution (%):")
print(df["y"].value_counts(normalize=True).round(3))


Shape: (4119, 21)

Column names:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

First five rows:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no



Missing values by column:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Target class distribution:
y
no     3668
yes     451
Name: count, dtype: int64

Target class distribution (%):
y
no     0.891
yes    0.109
Name: proportion, dtype: float64


## 4. Data Cleaning Checks

Before modeling, I check for duplicate rows and missing values. The dataset is mostly clean because it does not contain regular missing values. Some categorical values may be stored as `unknown`, but those are treated as valid categories in this project.


In [4]:
duplicate_count = df.duplicated().sum()
missing_count = df.isna().sum().sum()

print("Duplicate rows:", duplicate_count)
print("Total missing values:", missing_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates were removed. New shape:", df.shape)
else:
    print("No duplicate rows were found.")

if missing_count == 0:
    print("The dataset has no regular missing values, so no imputation is needed.")


Duplicate rows: 0
Total missing values: 0
No duplicate rows were found.
The dataset has no regular missing values, so no imputation is needed.


## 5. Feature Engineering

The Streamlit app uses several engineered features. I recreate those features here so the training pipeline uses the same type of inputs as the app. I also remove `duration` because it is only known after a phone call ends, so using it would cause data leakage.


In [5]:
def add_engineered_features(data):
    data = data.copy()

    data["has_university_degree"] = (data["education"] == "university.degree").astype(int)
    data["is_high_campaign"] = (data["campaign"] > 3).astype(int)

    data["white_collar"] = data["job"].isin(["admin.", "technician", "management"]).astype(int)
    data["blue_collar"] = data["job"].isin(["blue-collar", "services", "housemaid"]).astype(int)
    data["other_collar"] = data["job"].isin([
        "retired", "self-employed", "entrepreneur", "unemployed", "student"
    ]).astype(int)

    data["econ_interact"] = data["euribor3m"] * data["emp.var.rate"]
    data["age_euribor_interact"] = data["age"] * data["euribor3m"]
    data["campaign_conf_interact"] = data["campaign"] * data["cons.conf.idx"]
    data["cpi_euribor_diff"] = data["cons.price.idx"] - data["euribor3m"]

    data["previous_contact"] = (data["pdays"] != 999).astype(int)
    data["has_multiple_loans"] = ((data["housing"] == "yes") & (data["loan"] == "yes")).astype(int)
    data["economic_stress"] = (data["emp.var.rate"] < 0).astype(int)

    if "marital" in data.columns:
        data["married"] = (data["marital"] == "married").astype(int)

    return data

model_df = add_engineered_features(df)

if "duration" in model_df.columns:
    model_df = model_df.drop(columns=["duration"])

model_df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,blue_collar,other_collar,econ_interact,age_euribor_interact,campaign_conf_interact,cpi_euribor_diff,previous_contact,has_multiple_loans,economic_stress,married
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,1,0,-2.3634,39.390,-92.4,91.580,0,0,1,1
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,1,0,5.3405,189.345,-145.6,89.139,0,0,0,0
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,0,6.9468,124.050,-41.8,89.503,0,0,0,1
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,1,0,6.9426,188.442,-125.4,89.506,0,0,0,1
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,0,0,-0.4191,196.977,-42.0,89.009,0,0,1,1


## 6. Split X and y

The target `y` is converted from `yes` and `no` into 1 and 0. I also keep the features compatible with the Streamlit app, so columns not collected by the app are not used in the final model.


In [6]:
y = model_df["y"].map({"yes": 1, "no": 0})
X = model_df.drop(columns=["y"])

# These columns are not entered in the Streamlit app, so they are removed from the training inputs.
app_unused_columns = ["default", "day_of_week", "marital"]
X = X.drop(columns=[col for col in app_unused_columns if col in X.columns])

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())


X shape: (4119, 29)
y distribution:
y
0    3668
1     451
Name: count, dtype: int64


## 7. Train/Test Split

I use a stratified train/test split so the class balance stays similar in both sets.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("Test target distribution:")
print(y_test.value_counts(normalize=True).round(3))


Training shape: (3295, 29)
Test shape: (824, 29)
Training target distribution:
y
0    0.89
1    0.11
Name: proportion, dtype: float64
Test target distribution:
y
0    0.891
1    0.109
Name: proportion, dtype: float64


## 8. Preprocessing

Numerical columns are scaled with `StandardScaler`. Categorical columns are converted with `OneHotEncoder`. This lets the models use both numeric and text-based features correctly.


In [8]:
numerical_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)


Numerical features: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'has_university_degree', 'is_high_campaign', 'white_collar', 'blue_collar', 'other_collar', 'econ_interact', 'age_euribor_interact', 'campaign_conf_interact', 'cpi_euribor_diff', 'previous_contact', 'has_multiple_loans', 'economic_stress', 'married']
Categorical features: ['job', 'education', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 9. Feature Selection

I use `SelectKBest` with mutual information. This is a simple feature selection method that keeps the most useful encoded features for predicting the target.


In [9]:
selector = SelectKBest(score_func=mutual_info_classif, k=25)


## 10. Train and Compare Models

I compare Logistic Regression, Random Forest, and Gradient Boosting. Because the target classes are imbalanced, I include SMOTE inside the pipeline after preprocessing and feature selection.


In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = ImbPipeline(steps=[
        ("preprocessing", preprocessor),
        ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=25)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc,
    })

    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
results_df


,model,accuracy,precision,recall,f1,roc_auc
2,Gradient Boosting,0.889563,0.493506,0.422222,0.455090,0.773479
0,Logistic Regression,0.819175,0.323353,0.600000,0.420233,0.765070
1,Random Forest,0.890777,0.500000,0.333333,0.400000,0.762269


## 11. Detailed Metrics for the Best Initial Model

The table gives a quick comparison. Here I print the confusion matrix and classification report for the best initial model based on F1-score.


In [11]:
best_initial_name = results_df.iloc[0]["model"]
best_initial_pipeline = fitted_pipelines[best_initial_name]
y_pred_best = best_initial_pipeline.predict(X_test)

print("Best initial model:", best_initial_name)
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_best))
print("\nClassification report:")
print(classification_report(y_test, y_pred_best))


Best initial model: Gradient Boosting

Confusion matrix:
[[695  39]
 [ 52  38]]

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       734
           1       0.49      0.42      0.46        90

    accuracy                           0.89       824
   macro avg       0.71      0.68      0.70       824
weighted avg       0.88      0.89      0.89       824



## 12. Hyperparameter Tuning

Random Forest is a strong and easy-to-explain model for this project, so I tune it with `GridSearchCV`. The grid is kept small so the notebook can run in a reasonable amount of time.


In [12]:
rf_pipeline = ImbPipeline(steps=[
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=25)),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_grid = {
    "feature_selection__k": [20, 25, 30],
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_split": [2, 5],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("Best cross-validation F1:", round(grid_search.best_score_, 4))


Best parameters:
{'feature_selection__k': 30, 'model__max_depth': 8, 'model__min_samples_split': 5, 'model__n_estimators': 150}
Best cross-validation F1: 0.4701


## 13. Final Model Evaluation

After tuning, I evaluate the final Random Forest pipeline on the test set using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrix, and classification report.


In [13]:
final_pipeline = grid_search.best_estimator_
y_pred_final = final_pipeline.predict(X_test)
y_proba_final = final_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred_final), 4))
print("Precision:", round(precision_score(y_test, y_pred_final), 4))
print("Recall:", round(recall_score(y_test, y_pred_final), 4))
print("F1-score:", round(f1_score(y_test, y_pred_final), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_final), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification report:")
print(classification_report(y_test, y_pred_final))


Accuracy: 0.8823
Precision: 0.4639
Recall: 0.5
F1-score: 0.4813
ROC-AUC: 0.7806

Confusion matrix:
[[682  52]
 [ 45  45]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.93       734
           1       0.46      0.50      0.48        90

    accuracy                           0.88       824
   macro avg       0.70      0.71      0.71       824
weighted avg       0.89      0.88      0.88       824



## 14. Save the Final Pipeline

The final pipeline includes preprocessing, feature selection, SMOTE, and the tuned Random Forest model. It is saved as `rf_pipeline.sav`, which is the file used by the Streamlit app.


In [14]:
model_path = Path("rf_pipeline.sav")

with open(model_path, "wb") as f:
    pickle.dump(final_pipeline, f)

print(f"Final pipeline saved to: {model_path.resolve()}")


Final pipeline saved to: C:\Users\yanku\OneDrive\Masaüstü\ada\ADA\ADA-PROJECT-FINAL\rf_pipeline.sav


## 15. Conclusion

This notebook prepared the bank marketing dataset, removed the leakage column, added app-style engineered features, compared three classification models, tuned a Random Forest model, and saved the final pipeline for use in the Streamlit app.
